## locating files and setting up stuff

In [39]:
import os
os.chdir("/Users/asterchrome/Documents/RCS")
print("Working folder:", os.getcwd())
print("Data exists:", os.path.exists("data/amazon_sample.csv.gz"))

Working folder: /Users/asterchrome/Documents/RCS
Data exists: True


In [40]:
import re
import spacy
import pandas as pd
from tqdm.auto import tqdm

SEED = 42   # makes the random sampling repeatable means same seed = same rows every time

# Categories we set aside completely so that the the model never trains on these)
HOLDOUT_CATEGORIES = ["Books", "Software", "Automotive", "Pet_Supplies"]

# For each of the other 29 categories we have a small number of rows which we split into train, validation test sets
TRAIN_ROWS = 280   
VALID_ROWS = 35    
TEST_ROWS = 35     
ROWS_PER_SEEN_CATEGORY = TRAIN_ROWS + VALID_ROWS + TEST_ROWS   # = 350

#OOD rows for each of the 4 unseen categories
ROWS_PER_HOLDOUT_CATEGORY = 500

## Step 2 Data Loading and visualising

In [41]:
df = pd.read_csv("data/amazon_sample.csv.gz")
print("Rows and columns", df.shape)
print()
print("Reviews per category-")
print(df["category"].value_counts())

Rows and columns (815439, 9)

Reviews per category-
category
Books                          24880
Pet_Supplies                   24877
Beauty_and_Personal_Care       24864
Clothing_Shoes_and_Jewelry     24841
Tools_and_Home_Improvement     24822
Arts_Crafts_and_Sewing         24813
Grocery_and_Gourmet_Food       24809
Musical_Instruments            24801
Industrial_and_Scientific      24793
Software                       24789
Sports_and_Outdoors            24788
Kindle_Store                   24784
Video_Games                    24767
Appliances                     24761
Electronics                    24734
Movies_and_TV                  24734
Automotive                     24673
Health_and_Household           24639
Patio_Lawn_and_Garden          24633
Office_Products                24621
Home_and_Kitchen               24607
Cell_Phones_and_Accessories    24602
CDs_and_Vinyl                  24556
Toys_and_Games                 24547
Unknown                        24520
Amazon_Fashion

## Step 3 Removeing bad rows
- **Unknown category** -we dont know wth it is so we skip..
- **Missing text or rating** nothing to learn from...
- **Reviews under 3 words** like 'good' or 'bad' - too short so we skip
- **Duplicate texts** we dont repeat ourselves 

In [42]:
rows_before = len(df)

df = df[df["category"] != "Unknown"]
df = df.dropna(subset=["text", "rating"])
df["title"] = df["title"].fillna("")
df["rating"] = df["rating"].astype(int)

df["word_count"] = df["text"].str.split().str.len()
df = df[df["word_count"] >= 3]

df = df.drop_duplicates(subset=["text"])

print("Rows before cleaning-", rows_before)
print("Rows after cleaning- ", len(df))

Rows before cleaning- 815439
Rows after cleaning-  711419


## Step 4 Setting aside the 4 unseen categories for OOD wont be used for training
Later we also test the model on them: if it still does well, it has learned something general about reviews and not just memorised product types.

In [43]:
unseen_df = df[df["category"].isin(HOLDOUT_CATEGORIES)]
seen_df = df[~df["category"].isin(HOLDOUT_CATEGORIES)]

print("Unseen categories. ", unseen_df["category"].unique())
print("Seen categories.  ", seen_df["category"].nunique(), "categories")

Unseen categories.  <StringArray>
['Automotive', 'Books', 'Pet_Supplies', 'Software']
Length: 4, dtype: str
Seen categories.   29 categories


## Step 5 Split the 29 seen categories into train / validation / test


In [44]:
train_parts = []
valid_parts = []
test_parts = []

for category in sorted(seen_df["category"].unique()):
    one_category = seen_df[seen_df["category"] == category]

    # pick 350 random reviews from this category
    picked = one_category.sample(n=ROWS_PER_SEEN_CATEGORY, random_state=SEED)

    # cut the 350 into three groups
    train_parts.append(picked.iloc[0 : TRAIN_ROWS])
    valid_parts.append(picked.iloc[TRAIN_ROWS : TRAIN_ROWS + VALID_ROWS])
    test_parts.append(picked.iloc[TRAIN_ROWS + VALID_ROWS : ROWS_PER_SEEN_CATEGORY])

train = pd.concat(train_parts)
valid = pd.concat(valid_parts)
test = pd.concat(test_parts)

## Step 6 Build the unseen categories test set
Same idea, but these 4 categories are used **only for testing**.

In [45]:
unseen_parts = []

for category in HOLDOUT_CATEGORIES:
    one_category = unseen_df[unseen_df["category"] == category]
    unseen_parts.append(one_category.sample(n=ROWS_PER_HOLDOUT_CATEGORY, random_state=SEED))

unseen_test = pd.concat(unseen_parts)

## Step 7 Check that the split is what we claimed
Never trust a split without checking it. These checks print the evidence.

In [46]:
# Put the four groups in one dictionary so we can loop over them
splits = {"train": train, "valid": valid, "test": test, "unseen_test": unseen_test}

# Check 1 how many rows and categories in each group
print("CHECK 1: size of each split")
for name, data in splits.items():
    print(f"  {name:12s} {len(data):6d} rows, {data['category'].nunique():2d} categories")

total = sum(len(data) for data in splits.values())
print("  TOTAL:", total)
assert total < 13000, "Too many rows!"

# Check 2 train and unseen categories must not overlap
overlap = set(train["category"]) & set(unseen_test["category"])
print("\ncategories in both train and unseen_test:", overlap if overlap else "none (good)")

# Check 3 no identical review text in two different splits
print("\nidentical review texts shared between splits")
names = list(splits.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        shared = set(splits[names[i]]["text"]) & set(splits[names[j]]["text"])
        print(f"  {names[i]} and {names[j]}: {len(shared)}")

# Check 4 every seen category has the same number of rows in train
counts = train["category"].value_counts()
print("\nrows per category in train -> smallest:", counts.min(), ", largest:", counts.max())

CHECK 1: size of each split
  train          8120 rows, 29 categories
  valid          1015 rows, 29 categories
  test           1015 rows, 29 categories
  unseen_test    2000 rows,  4 categories
  TOTAL: 12150

categories in both train and unseen_test: none (good)

identical review texts shared between splits
  train and valid: 0
  train and test: 0
  train and unseen_test: 0
  valid and test: 0
  valid and unseen_test: 0
  test and unseen_test: 0

rows per category in train -> smallest: 280 , largest: 280


### Rating balance (star ratings 1 to 5)
We did **not** change the rating mix. Real Amazon reviews are mostly 5-star, so every split keeps that real mix.
This is honest, but it means a model could score well just by guessing "5" every time.
So when we train later, we will measure with **macro-F1** (which treats each star rating equally) and/or use class weights.

In [47]:
rating_table = pd.DataFrame({
    name: data["rating"].value_counts(normalize=True).sort_index().round(3)
    for name, data in splits.items()
})
print("Share of each star rating in every split:")
print(rating_table)

Share of each star rating in every split:
        train  valid   test  unseen_test
rating                                  
1       0.111  0.124  0.110        0.102
2       0.049  0.059  0.045        0.050
3       0.073  0.082  0.092        0.072
4       0.113  0.112  0.120        0.144
5       0.654  0.623  0.633        0.632


## Step 8 text cleaning
This is your original cleaning code, put into this notebook so everything lives in one file. 

makes everything lowercase
removes line breaks, HTML leftovers and special characters
replaces web links with the word LINK
splits the text into word pieces (tokens) with spaCy and joins them with spaces

In [48]:
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "ner", "lemmatizer", "attribute_ruler"])

# patterns to find junk in the text
HEX = re.compile(r'&#x[0-9A-Fa-f]+;')
DEC = re.compile(r'&#\d+;')
ASCII = re.compile(r'[^\x00-\x7F]+')
URL = re.compile(r'http\S+|www\S+')
ESCAPES = re.compile(r'[\n\t\r]+')

def clean_regex_only(text):
    x = str(text).lower()
    x = ESCAPES.sub(" ", x)
    x = x.replace("&amp;", "&").replace("<br />", '')
    x = HEX.sub('', x)
    x = DEC.sub("'", x)
    x = ASCII.sub("'", x)
    x = URL.sub('LINK', x)
    return x.strip()

def preprocess(text, nlp):
    clean_text = clean_regex_only(text)
    tokens = nlp.tokenizer(clean_text)
    return ' '.join([tok.text for tok in tokens])

def apply_preprocess(X):
    return [preprocess(sent, nlp) for sent in tqdm(X)]

In [49]:
for name in splits:
    print("Cleaning", name)
    data = splits[name].copy()
    data["text_clean"] = apply_preprocess(data["text"])
    data["title_clean"] = apply_preprocess(data["title"])
    splits[name] = data

# Look at a few before/after examples
example = splits["train"]
for i in range(3):
    print("\nORIGINAL:", example["text"].iloc[i][:150])
    print("CLEANED: ", example["text_clean"].iloc[i][:150])

Cleaning train


  0%|          | 0/8120 [00:00<?, ?it/s]

  0%|          | 0/8120 [00:00<?, ?it/s]

Cleaning valid


  0%|          | 0/1015 [00:00<?, ?it/s]

  0%|          | 0/1015 [00:00<?, ?it/s]

... (3 outputs left out)

Cleaning unseen_test


  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]


ORIGINAL: lashes look good
CLEANED:  lashes look good

ORIGINAL: Very manly smell. highly enjoyable product and looking forward to trying the other scents .
CLEANED:  very manly smell . highly enjoyable product and looking forward to trying the other scents .

ORIGINAL: Excellent product! Lots of brushes, so soft and lay down color beautifully.
CLEANED:  excellent product ! lots of brushes , so soft and lay down color beautifully .



saveing them into Each file where it keeps the original text and the cleaned text side by side.

In [50]:
os.makedirs("data/small", exist_ok=True)

columns_to_save = ["rating", "category", "title", "text", "title_clean", "text_clean"]

for name, data in splits.items():
    data[columns_to_save].to_csv(f"data/small/{name}.csv", index=False)
    print("saved", name, data.shape, "-> data/small/" + name + ".csv")

saved train (8120, 12) -> data/small/train.csv
saved valid (1015, 12) -> data/small/valid.csv
saved test (1015, 12) -> data/small/test.csv
saved unseen_test (2000, 12) -> data/small/unseen_test.csv
